# Model Inference — Walmart Store Sales Forecasting

This notebook takes the **champion** neural model (N-BEATS generic, validation
WMAE 2,193 — the best of all our neural experiments) and:

1. Wraps it as an MLflow pyfunc model that runs on the **raw** test set.
2. **Registers** it in the MLflow Model Registry on DagsHub.
3. **Loads it back from the registry** and predicts on `data/test.csv`.
4. Writes `submission.csv` in Kaggle format.

Champion model was saved by `model_experiment_NBEATS.ipynb` into
`saved_models/nbeats_nf` (retrained on the full history). N-BEATS is univariate, so it
only needs the past-sales history to forecast the next 43 weeks (which covers the whole
test period, 2012-11-02 to 2013-07-26).

In [1]:
# Run from the project root so data/ and src/ resolve.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
import dagshub
from mlflow.tracking import MlflowClient

from src.features.nn_preprocessing import build_long_df

warnings.filterwarnings("ignore")

# Experiment tracking: DagsHub-hosted MLflow.
dagshub.init(repo_owner="GigiSichinava", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("Inference")

# Champion config.
CHAMPION_DIR = "saved_models/nbeats_nf"
REGISTERED_NAME = "Walmart_Champion_NBEATS"
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as GigiSichinava

Initialized MLflow to track repo "GigiSichinava/Walmart-Sales-Forecasting"

Repository GigiSichinava/Walmart-Sales-Forecasting initialized!

2026/07/09 15:15:40 INFO mlflow.tracking.fluent: Experiment with name 'Inference' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow


In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
test_raw = pd.read_csv("data/test.csv")
print("test:", test_raw.shape, "| dates:", test_raw["Date"].min(), "->", test_raw["Date"].max())

test: (115064, 4) | dates: 2012-11-02 -> 2013-07-26


## Step 1 — Rebuild the sales history

The model forecasts from the full past-sales history. We rebuild it with the same
preprocessing used in training and save it as a small artifact that ships with the
registered model (so the model is self-contained).

In [4]:
history = build_long_df(train_raw, stores, features, fill_gaps=True)[["unique_id", "ds", "y"]]
os.makedirs("saved_models", exist_ok=True)
history.to_parquet("saved_models/history.parquet")
print("history rows:", len(history), "| series:", history["unique_id"].nunique())

history rows: 449237 | series: 3331


## Step 2 — Wrap the champion as an MLflow pyfunc model

The wrapper loads the saved N-BEATS model plus the history, and its `predict` takes the
**raw** test dataframe (`Store`, `Dept`, `Date`) and returns the weekly-sales forecast
for each row. This is the 'pipeline that runs on the un-preprocessed test set'.

In [5]:
class NBEATSForecaster(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import pandas as pd
        from neuralforecast import NeuralForecast
        self.nf = NeuralForecast.load(path=context.artifacts["nf_model"])
        self.history = pd.read_parquet(context.artifacts["history"])
        self.model_col = self.nf.models[0].alias  # "NBEATS"

    def predict(self, context, model_input):
        import pandas as pd
        test = model_input.copy()
        # Build the series id and date from the raw test columns.
        test["unique_id"] = test["Store"].astype(str) + "_" + test["Dept"].astype(str)
        test["ds"] = pd.to_datetime(test["Date"])
        # Forecast the next weeks from the stored history.
        fcst = self.nf.predict(df=self.history).rename(columns={self.model_col: "yhat"})
        out = test.merge(fcst[["unique_id", "ds", "yhat"]], on=["unique_id", "ds"], how="left")
        # Series unseen in training get 0; sales can't be negative.
        return out["yhat"].fillna(0).clip(lower=0).values

D:\Desktop\ML Project\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## Step 3 — Register the champion in the Model Registry

Logs the wrapped model to DagsHub and registers it under `Walmart_Champion_NBEATS`.
After this, it appears in the MLflow **Models** tab.

In [6]:
with mlflow.start_run(run_name="Register_Champion_NBEATS"):
    mlflow.log_param("champion", "NBEATS_generic")
    mlflow.log_metric("val_WMAE", 2193.87)

    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=NBEATSForecaster(),
        artifacts={"nf_model": CHAMPION_DIR, "history": "saved_models/history.parquet"},
        pip_requirements=["neuralforecast", "torch", "pandas", "numpy", "utilsforecast"],
        registered_model_name=REGISTERED_NAME,
    )
    print("Logged model URI:", model_info.model_uri)

2026/07/09 15:16:15 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Successfully registered model 'Walmart_Champion_NBEATS'.
2026/07/09 15:17:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_Champion_NBEATS, version 1
Created version '1' of model 'Walmart_Champion_NBEATS'.


Logged model URI: models:/m-f4240a82f38f4d04b848501059a39426
🏃 View run Register_Champion_NBEATS at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/4/runs/18e5e2e31b2e457382f92f8d4fcfa8cf
🧪 View experiment at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/4


## Step 4 — Load from the registry and predict on the raw test set

In [7]:
# Find the latest registered version and load it straight from the registry.
client = MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED_NAME}'")
latest = max(int(v.version) for v in versions)
print("Loading", REGISTERED_NAME, "version", latest)

champion = mlflow.pyfunc.load_model(f"models:/{REGISTERED_NAME}/{latest}")

preds = champion.predict(test_raw[["Store", "Dept", "Date", "IsHoliday"]])
print("predictions:", len(preds), "| min:", float(np.min(preds)), "| max:", float(np.max(preds)))

Loading Walmart_Champion_NBEATS version 1


2026-07-09 15:17:32,626	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-07-09 15:17:32,867	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|██████████| 105/105 [00:01<00:00, 79.16it/s]
predictions: 115064 | min: 0.0 | max: 185973.625


## Step 5 — Build the Kaggle submission

`Id` = `Store_Dept_Date`, matching `sampleSubmission.csv`.

In [8]:
submission = pd.DataFrame({
    "Id": test_raw["Store"].astype(str) + "_"
          + test_raw["Dept"].astype(str) + "_"
          + test_raw["Date"].astype(str),
    "Weekly_Sales": np.asarray(preds, dtype=float),
})
submission.to_csv("submission.csv", index=False)

print("rows:", len(submission), "| NaNs:", int(submission["Weekly_Sales"].isna().sum()))
print("negatives:", int((submission["Weekly_Sales"] < 0).sum()))
submission.head()

rows: 115064 | NaNs: 0
negatives: 0


,Id,Weekly_Sales
0,1_1_2012-11-02,27010.740234
1,1_1_2012-11-09,25191.371094
2,1_1_2012-11-16,26783.640625
3,1_1_2012-11-23,27194.824219
4,1_1_2012-11-30,28841.882812


## Notes

- Upload `submission.csv` to the Kaggle competition to get the public WMAE score —
  this is the unbiased test-set metric (no gap-fill zeros), so it is the number to
  report as the final result.
- The champion is loaded **from the Model Registry**, not from a local file, exactly as
  the assignment requires.
- To switch champions later, change `CHAMPION_DIR` / `REGISTERED_NAME` and re-run.